# CSC 2301 · Week 13 — Simulation: Monte Carlo oil in place

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/csc2301/notebooks/CSC2301_W13_monte_carlo_stoiip.ipynb)

**IUGB · BSc Mining / Petroleum Engineering** · Instructor: Laurent Kouadio

Before drilling, nobody knows the exact size of a reservoir: the area, the thickness and the porosity are
only known as **ranges**. A single "best guess" hides the risk. **Monte Carlo simulation** draws thousands
of possible reservoirs at random, computes the oil in place of each, and turns the results into the
figures every petroleum company reports: **P90, P50, P10**. The same method sizes ore bodies, schedules
projects and prices risk.

**What you will practise:** `np.random.default_rng` and a seed · uniform, normal and triangular draws ·
a vectorised model · percentiles · probabilities as averages of a mask · a histogram with markers.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/csc2301/'
FILES = [('tests/w13/q1.py', 'tests/q1.py'), ('tests/w13/q2.py', 'tests/q2.py'), ('tests/w13/q3.py', 'tests/q3.py'), ('tests/w13/q4.py', 'tests/q4.py'), ('tests/w13/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

## 1. Random numbers you can reproduce

A **generator** with a **seed** gives the same "random" numbers every time: your results can be checked.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
print(rng.uniform(0, 1, 3))          # 3 values, any value in [0, 1) equally likely
print(rng.normal(25, 4, 3))          # mean 25, standard deviation 4
print(rng.triangular(8, 12, 18, 3))  # min 8, most likely 12, max 18

In [ ]:
rng = np.random.default_rng(1)
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (name, x) in zip(axes, [("uniform(0.18, 0.26)", rng.uniform(0.18, 0.26, 20_000)),
                                ("normal(25, 4)", rng.normal(25, 4, 20_000)),
                                ("triangular(8, 12, 18)", rng.triangular(8, 12, 18, 20_000))]):
    ax.hist(x, bins=50)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 2. More draws, steadier answers

The mean of many random draws settles down as their number grows (law of large numbers).
That is why we use 100 000 draws, not 10.

In [ ]:
rng = np.random.default_rng(7)
for n in [10, 100, 1_000, 10_000, 100_000]:
    print(f"{n:>7} draws: mean of a uniform(0, 1) = {rng.uniform(0, 1, n).mean():.4f}")

## 3. The model

**Stock-tank oil initially in place**, in millions of barrels:

$$STOIIP = \frac{A \times 10^6 \times h \times \phi \times (1 - S_w)}{B_o} \times \frac{6.2898}{10^6}$$

$A$ area (km²), $h$ net thickness (m), $\phi$ porosity, $S_w$ water saturation, $B_o$ formation volume
factor (reservoir m³ per stock-tank m³), 6.2898 barrels per m³.

| parameter | distribution | values |
|---|---|---|
| area $A$ (km²) | triangular | min 8, most likely 12, max 18 |
| thickness $h$ (m) | normal, no value below 5 | mean 25, std 4 |
| porosity $\phi$ | uniform | 0.18 – 0.26 |
| water saturation $S_w$ | uniform | 0.20 – 0.40 |
| $B_o$ | fixed | 1.25 |

---
## Exercises

### Q1. The deterministic answer

Write `stoiip_mmbbl(area_km2, h_m, phi, sw, bo)` and store in `base_case` the STOIIP with the most likely
values (A = 12, h = 25, φ = 0.22, Sw = 0.30, Bo = 1.25).

In [ ]:
def stoiip_mmbbl(area_km2, h_m, phi, sw, bo):
    ...

base_case = ...
base_case

In [ ]:
grader.check("q1")

### Q2. Ready for arrays

If your function only uses `+ - * /`, it already works on NumPy arrays: try it below with two arrays.
Then create `rng = np.random.default_rng(13)` and store five uniform draws between 0 and 1 in `u`.

In [ ]:
print(stoiip_mmbbl(np.array([10.0, 14.0]), np.array([20.0, 30.0]), 0.22, 0.30, 1.25))

rng = ...
u = ...
u

In [ ]:
grader.check("q2")

### Q3. 100 000 possible reservoirs

With `rng` and `N = 100_000`, draw `area`, `h`, `phi` and `sw` following the table (clip `h` at 5 m with
`np.clip`), then compute `stoiip` for every draw in **one line**.

In [ ]:
N = 100_000
area = ...
h = ...
phi = ...
sw = ...
stoiip = ...
stoiip.mean(), stoiip.min(), stoiip.max()

In [ ]:
grader.check("q3")

### Q4. P90, P50, P10

The industry reports **P90** (a value exceeded with 90 % probability: the low, "proved-like" case),
**P50** (the median) and **P10** (the high case). P90 is therefore the **10th percentile**.
Store them in `p90`, `p50`, `p10` with one call to `np.percentile`. Is `base_case` close to `p50`?

In [ ]:
p90, p50, p10 = ...
print(f"P90 {p90:.0f}  P50 {p50:.0f}  P10 {p10:.0f} MMbbl   (base case {base_case:.0f})")

In [ ]:
grader.check("q4")

### Q5. Risk for the manager

The development is economic if STOIIP ≥ 200 MMbbl. Store in `prob_200` the probability of that (a
fraction). Then draw the histogram of `stoiip` in `fig_mc, ax_mc` (50 bins), with a vertical line at P90,
P50 and P10 (`ax_mc.axvline`), the x axis labelled `STOIIP (MMbbl)` and a legend.

In [ ]:
prob_200 = ...

fig_mc, ax_mc = plt.subplots(figsize=(7, 4))
...
plt.show()
prob_200

In [ ]:
grader.check("q5")

### Going further (not checked)

* Which parameter matters most? Replace one range at a time by its most likely value and see how much
  P10 − P90 shrinks (a "tornado" analysis).
* Run the simulation with 100 draws, five times with different seeds. How much does P50 move? And with 100 000?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()